# HLS Image Filter on AUP-ZU3 (Zynq UltraScale+ ZU3EG)

Load a JPEG, push it through a Sobel / grayscale accelerator built with Vitis HLS
running in the programmable logic, and display the result inline.

**Before running:** put `image_filter.bit` and `image_filter.hwh` in the same
directory as this notebook, plus a JPEG named `test.jpg`.

### A note on the Mali GPU

The ZU3EG *does* contain an ARM Mali-400 MP2 in the processing system, but it
plays no part in this flow, and it can't:

- **HLS targets the PL, not the PS.** Vitis HLS compiles C/C++ into RTL for the
  FPGA fabric. The Mali is a hardened block in the processor subsystem - there is
  no path from HLS to it.
- **Mali-400 has no compute API.** It supports OpenGL ES 1.1/2.0 and OpenVG 1.1
  only. No OpenCL, no compute shaders, so no general-purpose image processing.
- **Jupyter renders in your browser.** Anything the notebook displays is encoded
  on the board, sent over the network, and drawn by the client machine. The
  board's GPU and DisplayPort output are not in that path.

If you specifically need the Mali, that's a separate project: OpenGL ES rendering
to a DRM/KMS framebuffer on the PS DisplayPort, with a monitor attached. See the
README for what that involves.


## 1. Load the overlay

In [ ]:
import time
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

ol = Overlay("image_filter.bit")
ol.download()

filt = ol.image_filter_0
print(filt.register_map)

The register map tells you the exact names HLS generated. The 64-bit pointer
arguments are split into two 32-bit registers each — `src_1`/`src_2` and
`dst_1`/`dst_2`. If your names differ, adjust `run_filter` below to match.

Note the scalar arguments are `img_width`/`img_height`, not `width`/`height`.
That is deliberate. PYNQ builds this register map by creating a property per
register field on its `Register` class, and `Register.__init__` assigns
`self.width` — so a field named `width` shadows it with a property and the
next attribute read recurses forever:

```
RecursionError: maximum recursion depth exceeded
```

...raised by this very cell, before the accelerator is ever started. The
reserved field names are `address`, `width`, `debug` and `access`.


## 2. Load the JPEG

Decoding happens on the A53 cores via Pillow. Converting to RGBA gives 4 bytes per pixel, which matches the accelerator's packed 32-bit word.

In [ ]:
img = Image.open("test.jpg").convert("RGBA")

# Optional: cap the width at MAX_WIDTH from the HLS build.
MAX_WIDTH = 1920
if img.width > MAX_WIDTH:
    scale = MAX_WIDTH / img.width
    img = img.resize((MAX_WIDTH, int(img.height * scale)), Image.LANCZOS)

W, H = img.size
print(f"{W} x {H}  ({W*H/1e6:.2f} Mpixel)")
plt.figure(figsize=(8, 8 * H / W))
plt.imshow(img); plt.axis("off"); plt.title("input"); plt.show()

## 3. Allocate contiguous buffers

`allocate()` returns physically-contiguous DMA-capable memory and exposes its physical address, which is what the accelerator's AXI master needs.

In [ ]:
src_buf = allocate(shape=(H, W, 4), dtype=np.uint8)
dst_buf = allocate(shape=(H, W, 4), dtype=np.uint8)

src_buf[:] = np.array(img)
src_buf.flush()          # push out of the A53 caches so the PL sees current data

print(f"src physical address: 0x{src_buf.physical_address:012X}")
print(f"dst physical address: 0x{dst_buf.physical_address:012X}")

## 4. Drive the accelerator

In [ ]:
MODE_GRAY, MODE_SOBEL, MODE_INVERT = 0, 1, 2

def run_filter(mode, wait=True):
    rm = filt.register_map
    sp, dp = src_buf.physical_address, dst_buf.physical_address

    rm.src_1 = sp & 0xFFFFFFFF
    rm.src_2 = (sp >> 32) & 0xFFFFFFFF
    rm.dst_1 = dp & 0xFFFFFFFF
    rm.dst_2 = (dp >> 32) & 0xFFFFFFFF
    rm.img_width  = W
    rm.img_height = H
    rm.mode       = mode

    rm.CTRL.AP_START = 1                 # kick off
    if wait:
        while rm.CTRL.AP_DONE == 0:      # poll ap_done
            pass
        dst_buf.invalidate()             # drop stale cache lines before reading
    return dst_buf

t0 = time.perf_counter()
run_filter(MODE_SOBEL)
t_hw = time.perf_counter() - t0
print(f"PL Sobel: {t_hw*1000:.2f} ms  ({W*H/t_hw/1e6:.1f} Mpixel/s)")

## 5. Display the result

In [ ]:
result = np.array(dst_buf)

fig, ax = plt.subplots(1, 2, figsize=(14, 7 * H / W))
ax[0].imshow(img);    ax[0].set_title("input JPEG");        ax[0].axis("off")
ax[1].imshow(result); ax[1].set_title("PL Sobel (HLS IP)"); ax[1].axis("off")
plt.tight_layout(); plt.show()

## 6. All three modes

In [ ]:
modes = [(MODE_GRAY, "grayscale"), (MODE_SOBEL, "sobel"), (MODE_INVERT, "inverted")]

fig, ax = plt.subplots(1, 3, figsize=(16, 5.5 * H / W))
for a, (m, name) in zip(ax, modes):
    run_filter(m)
    a.imshow(np.array(dst_buf)); a.set_title(name); a.axis("off")
plt.tight_layout(); plt.show()

## 7. Check against a NumPy reference

Sanity check plus a rough sense of the speedup over software on the A53.

In [ ]:
def numpy_sobel(rgba):
    r = rgba[:, :, 0].astype(np.int32)
    g = rgba[:, :, 1].astype(np.int32)
    b = rgba[:, :, 2].astype(np.int32)
    gray = ((77 * r + 150 * g + 29 * b) >> 8).astype(np.int32)

    out = np.zeros_like(gray)
    p = gray
    gx = (p[:-2, 2:] + 2 * p[1:-1, 2:] + p[2:, 2:]) - (p[:-2, :-2] + 2 * p[1:-1, :-2] + p[2:, :-2])
    gy = (p[2:, :-2] + 2 * p[2:, 1:-1] + p[2:, 2:]) - (p[:-2, :-2] + 2 * p[:-2, 1:-1] + p[:-2, 2:])
    out[1:-1, 1:-1] = np.clip(np.abs(gx) + np.abs(gy), 0, 255)
    return out.astype(np.uint8)

rgba_in = np.array(img)

t0 = time.perf_counter()
ref = numpy_sobel(rgba_in)
t_sw = time.perf_counter() - t0

run_filter(MODE_SOBEL)
hw = np.array(dst_buf)[:, :, 0]

diff = np.abs(hw.astype(np.int16) - ref.astype(np.int16))
print(f"max abs difference : {diff.max()}")
print(f"pixels differing   : {int((diff != 0).sum())} of {W*H}")
print(f"NumPy on A53       : {t_sw*1000:.2f} ms")
print(f"PL accelerator     : {t_hw*1000:.2f} ms   ->  {t_sw/t_hw:.2f}x")

## 8. Free the buffers

Contiguous memory is a scarce resource - always release it.

In [ ]:
src_buf.freebuffer()
dst_buf.freebuffer()
print("done")